# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/data00077/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — Content lifecycle: The research paper reports that growing and declining pages have almost the same average word count, while declining pages are older on average. My methodology question is: how exactly was the growing/declining label defined, and are the comparisons descriptive associations rather than evidence that age causes decline? I would also check whether page age and other page characteristics could confound the comparison.

Finding 2 — Refreshing pages: The paper reports positive impression lift for refreshed pages in held-out strata. My methodology question is: how were refreshed and non-refreshed pages assigned, and does the validation design control for pre-existing differences between pages that were refreshed and pages that were not? I would want to know whether the evidence supports a directional association or a causal claim.

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before: I first evaluated the model with a row-level split. This can be optimistic because pages from the same client may appear in both training and testing.

After: I evaluated the same Random Forest with a client-grouped holdout, keeping clients entirely within either train or test. This is a more honest test of whether the model generalizes to unseen clients.

I compare Precision@20, Precision@50, Precision@100, ROC-AUC, and Average Precision. The difference between the two splits is treated as a validation-design effect, not as proof that one model is universally better.

In [42]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



# ML-09 — Before/After validation audit

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# Load the Week-5 feature vector
frame = pd.read_csv(
    "data/processed/refresh_feature_vector.csv"
)

TARGET = "is_declining_label"

MODEL_NUMERIC_FEATURES = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

# Keep only features that actually exist
features = [
    c for c in MODEL_NUMERIC_FEATURES
    if c in frame.columns
]

X = frame[features].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

y = frame[TARGET].astype(int)

# Client groups for honest validation
groups = frame["client_id"]


def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    return top_k.mean()


def evaluate_model(X_train, X_test, y_train, y_test, label):
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]

    results = {
        "Split": label,
        "Precision@20": precision_at_k(probabilities, y_test, 20),
        "Precision@50": precision_at_k(probabilities, y_test, 50),
        "Precision@100": precision_at_k(probabilities, y_test, 100),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Average Precision": average_precision_score(y_test, probabilities)
    }

    return results


# ---------------------------------------------------------
# BEFORE: ordinary row-level random split
# ---------------------------------------------------------

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

before = evaluate_model(
    X_train_random,
    X_test_random,
    y_train_random,
    y_test_random,
    "Before: random row split"
)


# ---------------------------------------------------------
# AFTER: client-grouped holdout
# ---------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

after = evaluate_model(
    X_train_grouped,
    X_test_grouped,
    y_train_grouped,
    y_test_grouped,
    "After: client-grouped holdout"
)


# ---------------------------------------------------------
# Show both results
# ---------------------------------------------------------

comparison = pd.DataFrame([before, after])

print("Features used:")
print(features)

print("\nRows:", len(frame))

print("\nValidation comparison:")
display(comparison.round(4))

print("\nTrain/test sizes:")
print(
    "Random split:",
    len(X_train_random),
    "train /",
    len(X_test_random),
    "test"
)

print(
    "Client-grouped:",
    len(X_train_grouped),
    "train /",
    len(X_test_grouped),
    "test"
)

# Verify that no client appears in both grouped train and test
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

overlap = train_clients.intersection(test_clients)

print("\nClient overlap in grouped split:", len(overlap))

if len(overlap) == 0:
    print("PASS: no client appears in both train and test.")
else:
    print("WARNING: client overlap detected.")

Features used:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']

Rows: 30000

Validation comparison:


,Split,Precision@20,Precision@50,Precision@100,ROC-AUC,Average Precision
0,Before: random row split,0.9,0.92,0.93,0.7513,0.7611
1,After: client-grouped holdout,0.8,0.72,0.66,0.5880,0.5929



Train/test sizes:
Random split: 24000 train / 6000 test
Client-grouped: 23837 train / 6163 test

Client overlap in grouped split: 0
PASS: no client appears in both train and test.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I checked the final feature set for variables that directly encode the outcome, identifiers, or information that would only be available after the prediction decision.

The target is `is_declining_label`, which is derived from `trend_direction`. Therefore, outcome-derived variables such as `trend_pct` and `trend_direction` must not be used as predictors.

I also excluded `content_id` and `client_id` from the model because they are identifiers rather than useful predictive signals and could allow the model to memorize entities.

The final model uses only these six observable features:

- `content_age_days`
- `days_since_last_update`
- `impressions_90d`
- `avg_position`
- `ctr`
- `word_count`

The leakage audit found no obvious leakage terms inside the final model feature set.

This follows the same principle demonstrated in the earlier exercise: `trend_pct` can nearly reproduce the label because the label is derived from the trend, so using it would make the evaluation misleading. :contentReference[oaicite:3]{index=3}

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-09 — Leakage audit

TARGET = "is_declining_label"

# Use the same Week-5 model features
MODEL_NUMERIC_FEATURES = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

# Build X explicitly
features = [
    c for c in MODEL_NUMERIC_FEATURES
    if c in frame.columns
]

X = frame[features].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

print("Potential leakage / identifier columns found:")

leakage_terms = [
    "trend",
    "label",
    "target",
    "outcome",
    "id"
]

leakage_columns = [
    c for c in frame.columns
    if any(term in c.lower() for term in leakage_terms)
]

print(leakage_columns)

print("\nTarget:")
print(TARGET)

print("\nModel features containing obvious leakage names:")
print([
    c for c in X.columns
    if any(
        term in c.lower()
        for term in ["trend", "label", "target", "outcome", "id"]
    )
])

print("\nFinal model features:")
print(X.columns.tolist())

print("\nLeakage audit complete.")

Potential leakage / identifier columns found:
['content_id', 'client_id', 'provider_used', 'trend_direction', 'trend_pct', 'is_declining_label']

Target:
is_declining_label

Model features containing obvious leakage names:
[]

Final model features:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']

Leakage audit complete.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

My original model results should not be presented as proof that the model will work universally.

A safer claim is:

"On the evaluated dataset and validation setup, the Random Forest showed measurable ability to distinguish pages labeled as declining from pages not labeled as declining."

The model provides directional decision-support evidence for prioritizing content for review. It does not prove that the model will generalize to every client or that any feature causes content decline.

The grouped client-holdout evaluation is a more honest estimate of performance on unseen clients than a simple row-level random split.

Final public-safe framing:

"The findings are observed and measured on this dataset and validation design. They provide directional decision-support evidence and should not be interpreted as universal predictions or causal evidence."

In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.


## Self-check

- [x] Two research-paper findings and constructive methodology questions are included.
- [x] The Week-5 Random Forest is evaluated with a random split and a client-grouped split.
- [x] Before/after metrics are calculated from the data rather than invented.
- [x] The grouped split checks that no client appears in both train and test.
- [x] The final feature set contains six observable features.
- [x] Leakage from trend-derived variables is explicitly discussed.
- [x] Claims use careful language such as observed, measured, directional, and decision-support.
- [x] No client names, private queries, or sensitive information are included.
- [ ] Run Runtime → Run all with no errors.
- [ ] Commit the executed notebook to `work/notebooks/w06_validation_audit.ipynb`.